# LearnGraph: Multi-Source Agentic RAG

LearnGraph is an educational AI agent that helps students learn AI-agent concepts and test their understanding using grounded course material.

**Modes**
1. **Learn Mode** — answers questions using retrieved learning content.
2. **Quiz Mode** — generates a question, pauses for the student's answer, then scores and gives feedback.

**Knowledge sources**
- **Microsoft AI Agents for Beginners** — conceptual topics
- **Official LangGraph documentation** — implementation topics (`@task`, `@entrypoint`, `interrupt()`)

This notebook follows **Track C: Multi-Source Knowledge Base with Routing**, building the pipeline from documents to an agentic RAG agent.


# Loading and Cleaning Course Documents

The retrieval pipeline starts here:

Markdown files → Cleaning → Documents → Chunks → Embeddings → Chroma → Retrieval → Grounded Answer

This section loads the Microsoft AI Agents course Markdown files, removes non-educational noise, and turns each file into a LangChain `Document` with metadata (title, lesson, source URL).


In [16]:
from pathlib import Path
from collections import Counter
import re
import shutil

from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

In [17]:
def find_project_root() -> Path:
    """
    Find the project root whether the notebook is running from:
    - learngraph/
    - learngraph/notebooks/
    """
    current_directory = Path.cwd().resolve()

    possible_roots = [
        current_directory,
        current_directory.parent,
    ]

    for possible_root in possible_roots:
        if (possible_root / "pyproject.toml").exists():
            return possible_root

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the notebook is inside the LearnGraph project."
    )


PROJECT_ROOT = find_project_root()
COURSE_DIR = PROJECT_ROOT / "data" / "course"
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore" / "course"

print("Project root:", PROJECT_ROOT)
print("Course directory:", COURSE_DIR)
print("Course directory exists:", COURSE_DIR.exists())

Project root: /Users/khawla/learngraph
Course directory: /Users/khawla/learngraph/data/course
Course directory exists: True


In [18]:
markdown_files = sorted(COURSE_DIR.rglob("*.md"))

print("Number of Markdown files:", len(markdown_files))

for file_path in markdown_files:
    relative_path = file_path.relative_to(COURSE_DIR)
    print("-", relative_path)

Number of Markdown files: 7
- microsoft-ai-agents/01-intro-to-ai-agents.md
- microsoft-ai-agents/03-agentic-design-patterns.md
- microsoft-ai-agents/04-tool-use.md
- microsoft-ai-agents/05-agentic-rag.md
- microsoft-ai-agents/07-planning-design.md
- microsoft-ai-agents/12-context-engineering.md
- microsoft-ai-agents/13-agent-memory.md


In [20]:
assert COURSE_DIR.exists(), "data/course does not exist."
assert len(markdown_files) > 0, "No Markdown files were found."

print("Course files found")

Course files found


In [98]:
def remove_non_educational_lines(text: str) -> str:
    excluded_phrases = [
        "click the image above to watch the video",
        "come say hi in the",
    ]

    kept_lines = []

    for line in text.splitlines():
        normalized_line = line.lower().strip()

        if any(
            phrase in normalized_line
            for phrase in excluded_phrases
        ):
            continue

        kept_lines.append(line)

    return "\n".join(kept_lines)


def clean_markdown(text: str) -> str:
    text = re.sub(
        r"\A---\s*\n.*?\n---\s*\n",
        "",
        text,
        flags=re.DOTALL,
    )

    text = re.sub(
        r"<!--.*?-->",
        "",
        text,
        flags=re.DOTALL,
    )

    text = re.sub(
        r"\[!\[[^\]]*\]\([^)]+\)\]\([^)]+\)",
        "",
        text,
    )

    text = re.sub(
        r"!\[[^\]]*\]\([^)]+\)",
        "",
        text,
    )

    text = re.sub(
        r"\[\s*\]\([^)]+\)",
        "",
        text,
    )

    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text,
    )

    text = re.sub(
        r"<a\b[^>]*>(.*?)</a>",
        r"\1",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    text = remove_non_educational_lines(text)

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


In [99]:
sample_path = markdown_files[0]
sample_text = sample_path.read_text(encoding="utf-8")
cleaned_sample = clean_markdown(sample_text)


# print("File:", sample_path.name)
# print()
# print(cleaned_sample[:1500])

In [100]:
from langchain_core.documents import Document

In [101]:
def extract_document_title(
    text: str,
    fallback_title: str,
) -> str:
    """Extract the first Markdown H1 title."""

    title_match = re.search(
        pattern=r"^#\s+(.+)$",
        string=text,
        flags=re.MULTILINE,
    )

    if title_match:
        return title_match.group(1).strip()

    return fallback_title

In [102]:
MICROSOFT_COURSE_BASE_URL = (
    "https://github.com/microsoft/"
    "ai-agents-for-beginners/blob/main"
)


def build_source_url(file_path: Path) -> str:
    """Build the original GitHub lesson URL."""

    lesson_folder = file_path.stem

    return (
        f"{MICROSOFT_COURSE_BASE_URL}/"
        f"{lesson_folder}/README.md"
    )

In [103]:
EXCLUDED_FILE_NAMES = {
    "LICENSE.md",
}


def load_course_documents(
    file_paths: list[Path],
) -> list[Document]:
    """
    Read, clean, and convert course Markdown files
    into LangChain Documents.
    """

    documents: list[Document] = []

    for file_path in file_paths:
        if file_path.name in EXCLUDED_FILE_NAMES:
            continue

        raw_text = file_path.read_text(
            encoding="utf-8"
        )

        cleaned_text = clean_markdown(raw_text)

        # Skip empty or extremely short files
        if len(cleaned_text) < 200:
            print(
                f"Skipped short file: {file_path.name}"
            )
            continue

        title = extract_document_title(
            text=cleaned_text,
            fallback_title=file_path.stem,
        )

        relative_path = file_path.relative_to(
            COURSE_DIR
        )

        document = Document(
            page_content=cleaned_text,
            metadata={
                "title": title,
                "lesson": file_path.stem,
                "file_name": file_path.name,
                "relative_path": str(relative_path),
                "source_type": "microsoft_ai_agents_course",
                "source_url": build_source_url(file_path),
            },
        )

        documents.append(document)

    return documents

In [104]:
raw_documents = load_course_documents(
    markdown_files
)

print(
    "Number of loaded documents:",
    len(raw_documents),
)


Number of loaded documents: 7


In [105]:
first_document = raw_documents[0]

print("Metadata:")
print(first_document.metadata)

print("\nContent preview:")
print(first_document.page_content[:1200])

Metadata:
{'title': 'Introduction to AI Agents and Agent Use Cases', 'lesson': '01-intro-to-ai-agents', 'file_name': '01-intro-to-ai-agents.md', 'relative_path': 'microsoft-ai-agents/01-intro-to-ai-agents.md', 'source_type': 'microsoft_ai_agents_course', 'source_url': 'https://github.com/microsoft/ai-agents-for-beginners/blob/main/01-intro-to-ai-agents/README.md'}

Content preview:
# Introduction to AI Agents and Agent Use Cases

Welcome to the **AI Agents for Beginners** course! This course gives you the foundational knowledge — and real working code — to start building AI Agents from scratch.

Before we jump into building, let's make sure we actually understand what an AI Agent *is* and when it makes sense to use one.

---

## Introduction

This lesson covers:

- What AI Agents are, and the different types that exist
- Which kinds of tasks AI Agents are best suited for
- The core building blocks you'll use when designing an Agentic solution

## Learning Goals

By the end of this less

In [106]:
for index, document in enumerate(
    raw_documents,
    start=1,
):
    print(
        f"{index}. "
        f"{document.metadata['title']}"
    )

    print(
        f"   Lesson: "
        f"{document.metadata['lesson']}"
    )

    print(
        f"   Characters: "
        f"{len(document.page_content):,}"
    )

    print()

1. Introduction to AI Agents and Agent Use Cases
   Lesson: 01-intro-to-ai-agents
   Characters: 6,794

2. AI Agentic Design Principles
   Lesson: 03-agentic-design-patterns
   Characters: 6,822

3. Tool Use Design Pattern
   Lesson: 04-tool-use
   Characters: 17,222

4. Agentic RAG
   Lesson: 05-agentic-rag
   Characters: 14,611

5. Planning Design
   Lesson: 07-planning-design
   Characters: 11,873

6. Context Engineering for AI Agents
   Lesson: 12-context-engineering
   Characters: 14,310

7. Memory for AI Agents
   Lesson: 13-agent-memory
   Characters: 11,950



# Chunking the Course Documents

In this section, the cleaned course documents are split into smaller retrieval units called **chunks**.

The splitting process happens in two stages:

1. Split each Markdown document by its headings.
2. Split long sections into smaller overlapping chunks.

This preserves the lesson structure while keeping each chunk small enough for semantic search and RAG.

In [107]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

In [108]:
HEADERS_TO_SPLIT_ON = [
    ("#", "heading_1"),
    ("##", "heading_2"),
    ("###", "heading_3"),
    ("####", "heading_4"),
]

markdown_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT_ON,
    strip_headers=False,
)

In [109]:
def split_documents_by_headers(
    documents: list[Document],
) -> list[Document]:
    """
    Split course documents by Markdown headings
    while preserving the original document metadata.
    """

    sections: list[Document] = []

    for document in documents:
        document_sections = (
            markdown_header_splitter.split_text(
                document.page_content
            )
        )

        for section in document_sections:
            section.metadata = {
                **document.metadata,
                **section.metadata,
            }

            sections.append(section)

    return sections

In [110]:
header_sections = split_documents_by_headers(
    raw_documents
)

print("Documents before splitting:", len(raw_documents))
print("Sections after heading splitting:", len(header_sections))

Documents before splitting: 7
Sections after heading splitting: 118


In [111]:
sample_section = header_sections[0]

print("Metadata:")
print(sample_section.metadata)

print("\nContent:")
print(sample_section.page_content[:1200])

Metadata:
{'title': 'Introduction to AI Agents and Agent Use Cases', 'lesson': '01-intro-to-ai-agents', 'file_name': '01-intro-to-ai-agents.md', 'relative_path': 'microsoft-ai-agents/01-intro-to-ai-agents.md', 'source_type': 'microsoft_ai_agents_course', 'source_url': 'https://github.com/microsoft/ai-agents-for-beginners/blob/main/01-intro-to-ai-agents/README.md', 'heading_1': 'Introduction to AI Agents and Agent Use Cases'}

Content:
# Introduction to AI Agents and Agent Use Cases  
Welcome to the **AI Agents for Beginners** course! This course gives you the foundational knowledge — and real working code — to start building AI Agents from scratch.  
Before we jump into building, let's make sure we actually understand what an AI Agent *is* and when it makes sense to use one.  
---


## Creating the Final Chunks

The Markdown sections are now divided into smaller overlapping chunks.

This step ensures that:

- Each chunk is small enough for semantic retrieval.
- Paragraphs and sentences remain together when possible.
- A small overlap preserves context between neighboring chunks.
- Lesson and heading metadata remain attached to every chunk.

In [112]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        "",
    ],
    add_start_index=True,
)

In [113]:
chunks = recursive_splitter.split_documents(
    header_sections
)

print("Header sections:", len(header_sections))
print("Final chunks:", len(chunks))

Header sections: 118
Final chunks: 149


In [114]:
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = (
        f"course-chunk-{index:04d}"
    )

print("First chunk ID:", chunks[0].metadata["chunk_id"])
print("Last chunk ID:", chunks[-1].metadata["chunk_id"])

First chunk ID: course-chunk-0000
Last chunk ID: course-chunk-0148


In [115]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Number of chunks:", len(chunks))
print("Smallest chunk:", min(chunk_lengths))
print("Largest chunk:", max(chunk_lengths))

print(
    "Average chunk length:",
    round(
        sum(chunk_lengths)
        / len(chunk_lengths)
    ),
)

Number of chunks: 149
Smallest chunk: 28
Largest chunk: 1200
Average chunk length: 561


In [ ]:
#sample
middle_index = len(chunks) // 2  
middle_chunk = chunks[middle_index]

print("Chunk index:", middle_index)

print("\nMetadata:")
for key, value in middle_chunk.metadata.items():
    print(f"{key}: {value}")

print("\nContent:")
print(middle_chunk.page_content)

Chunk index: 74

Metadata:
title: Agentic RAG
lesson: 05-agentic-rag
file_name: 05-agentic-rag.md
relative_path: microsoft-ai-agents/05-agentic-rag.md
source_type: microsoft_ai_agents_course
source_url: https://github.com/microsoft/ai-agents-for-beginners/blob/main/05-agentic-rag/README.md
heading_1: Agentic RAG
heading_2: Boundaries of Agency
start_index: 0
chunk_id: course-chunk-0074

Content:
## Boundaries of Agency  
Despite its autonomy within a task, Agentic RAG is not analogous to Artificial General Intelligence. Its “agentic” capabilities are confined to the tools, data sources, and policies provided by human developers. It can’t invent its own tools or step outside the domain boundaries that have been set. Rather, it excels at dynamically orchestrating the resources at hand.
Key differences from more advanced AI forms include:  
1. **Domain-Specific Autonomy:** Agentic RAG systems are focused on achieving user-defined goals within a known domain, employing strategies like quer

In [122]:
smallest_chunk_index = chunk_lengths.index(
    min(chunk_lengths)
)

largest_chunk_index = chunk_lengths.index(
    max(chunk_lengths)
)

smallest_chunk = chunks[smallest_chunk_index]
largest_chunk = chunks[largest_chunk_index]


print("SMALLEST CHUNK")
print("Index:", smallest_chunk_index)
print("Length:", len(smallest_chunk.page_content))
print("Metadata:", smallest_chunk.metadata)
print("Content:")
print(repr(smallest_chunk.page_content))



SMALLEST CHUNK
Index: 60
Length: 28
Metadata: {'title': 'Tool Use Design Pattern', 'lesson': '04-tool-use', 'file_name': '04-tool-use.md', 'relative_path': 'microsoft-ai-agents/04-tool-use.md', 'source_type': 'microsoft_ai_agents_course', 'source_url': 'https://github.com/microsoft/ai-agents-for-beginners/blob/main/04-tool-use/README.md', 'heading_1': 'Tool Use Design Pattern', 'heading_2': 'Next Lesson', 'start_index': 0, 'chunk_id': 'course-chunk-0060'}
Content:
'## Next Lesson  \nAgentic RAG'


In [123]:
print("LARGEST CHUNK")
print("Index:", largest_chunk_index)
print("Length:", len(largest_chunk.page_content))
print("Metadata:", largest_chunk.metadata)
print("Content:")
print(largest_chunk.page_content)

LARGEST CHUNK
Index: 50
Length: 1200
Metadata: {'title': 'Tool Use Design Pattern', 'lesson': '04-tool-use', 'file_name': '04-tool-use.md', 'relative_path': 'microsoft-ai-agents/04-tool-use.md', 'source_type': 'microsoft_ai_agents_course', 'source_url': 'https://github.com/microsoft/ai-agents-for-beginners/blob/main/04-tool-use/README.md', 'heading_1': 'Tool Use Design Pattern', 'heading_2': 'Tool Use Examples with Agentic Frameworks', 'heading_3': 'Microsoft Foundry Agent Service', 'start_index': 0, 'chunk_id': 'course-chunk-0050'}
Content:
### Microsoft Foundry Agent Service  
Microsoft Foundry Agent Service is a newer agentic framework that is designed to empower developers to securely build, deploy, and scale high-quality, and extensible AI agents without needing to manage the underlying compute and storage resources. It is particularly useful for enterprise applications since it is a fully managed service with enterprise grade security.  
When compared to developing with the LLM A

In [124]:
very_small_chunks = [
    chunk
    for chunk in chunks
    if len(chunk.page_content.strip()) < 50
]

print(
    "Chunks smaller than 50 characters:",
    len(very_small_chunks),
)

Chunks smaller than 50 characters: 10


In [125]:
for chunk in very_small_chunks:
    print("=" * 70)
    print("ID:", chunk.metadata["chunk_id"])
    print("Lesson:", chunk.metadata.get("lesson"))
    print("Content:", repr(chunk.page_content))

ID: course-chunk-0014
Lesson: 01-intro-to-ai-agents
Content: '## Previous Lesson  \nCourse Setup'
ID: course-chunk-0015
Lesson: 01-intro-to-ai-agents
Content: '## Next Lesson  \nExploring Agentic Frameworks'
ID: course-chunk-0030
Lesson: 03-agentic-design-patterns
Content: '## Previous Lesson  \nExploring Agentic Frameworks'
ID: course-chunk-0031
Lesson: 03-agentic-design-patterns
Content: '## Next Lesson  \nTool Use Design Pattern'
ID: course-chunk-0060
Lesson: 04-tool-use
Content: '## Next Lesson  \nAgentic RAG'
ID: course-chunk-0082
Lesson: 05-agentic-rag
Content: '## Previous Lesson  \nTool Use Design Pattern'
ID: course-chunk-0083
Lesson: 05-agentic-rag
Content: '## Next Lesson  \nBuilding Trustworthy AI Agents'
ID: course-chunk-0102
Lesson: 07-planning-design
Content: '## Next Lesson  \nMulti-Agent Design Pattern'
ID: course-chunk-0125
Lesson: 12-context-engineering
Content: '## Previous Lesson  \nAgentic Protocols'
ID: course-chunk-0126
Lesson: 12-context-engineering
Content: '#

In [127]:
NOISE_ONLY_CONTENT = {
    "---",
    "***",
    "___",
}


def is_useful_chunk(chunk: Document) -> bool:
    content = chunk.page_content.strip()

    if not content:
        return False

    if content in NOISE_ONLY_CONTENT:
        return False

    return True



chunks_before_filtering = len(chunks)

chunks = [
    chunk
    for chunk in chunks
    if is_useful_chunk(chunk)
]

print(
    "Chunks before filtering:",
    chunks_before_filtering,
)

print(
    "Chunks after filtering:",
    len(chunks),
)

Chunks before filtering: 149
Chunks after filtering: 149


In [128]:
chunks_without_headers = [
    chunk
    for chunk in chunks
    if not any(
        key.startswith("heading_")
        for key in chunk.metadata
    )
]

print(
    "Chunks without heading metadata:",
    len(chunks_without_headers),
)

Chunks without heading metadata: 4


In [129]:
for chunk in chunks_without_headers[:5]:
    print("=" * 70)
    print("ID:", chunk.metadata["chunk_id"])
    print("Lesson:", chunk.metadata.get("lesson"))
    print("Content:")
    print(chunk.page_content[:500])

ID: course-chunk-0016
Lesson: 03-agentic-design-patterns
Content:
> _(Click the image above to view video of this lesson)_
ID: course-chunk-0032
Lesson: 04-tool-use
Content:
> _(Click the image above to view video of this lesson)_
ID: course-chunk-0061
Lesson: 05-agentic-rag
Content:
> _(Click the image above to view video of this lesson)_
ID: course-chunk-0084
Lesson: 07-planning-design
Content:
> _(Click the image above to view video of this lesson)_


In [131]:
print("SMALLEST CHUNK")
print("Index:", smallest_chunk_index)
print("Length:", len(smallest_chunk.page_content))
print("Chunk ID:", smallest_chunk.metadata.get("chunk_id"))
print("Lesson:", smallest_chunk.metadata.get("lesson"))

print(
    "Section:",
    smallest_chunk.metadata.get("heading_4")
    or smallest_chunk.metadata.get("heading_3")
    or smallest_chunk.metadata.get("heading_2")
    or smallest_chunk.metadata.get("heading_1")
)

print("\nContent:")
print(repr(smallest_chunk.page_content))

SMALLEST CHUNK
Index: 60
Length: 28
Chunk ID: course-chunk-0060
Lesson: 04-tool-use
Section: Next Lesson

Content:
'## Next Lesson  \nAgentic RAG'


In [ ]:
NAVIGATION_HEADINGS = {
    "next lesson",
    "previous lesson",
}

NOISE_ONLY_CONTENT = {
    "---",
    "***",
    "___",
}


def normalize_value(value: str) -> str:
    """Normalize text for reliable comparison."""
    return " ".join(value.lower().strip().split())


def is_useful_chunk(chunk: Document) -> bool:
    """Return False for empty, separator, or navigation chunks."""

    content = chunk.page_content.strip()

    # Remove empty chunks
    if not content:
        return False

    # Remove chunks containing only Markdown separators
    if content in NOISE_ONLY_CONTENT:
        return False

    # Collect available Markdown headings
    headings = [
        chunk.metadata.get(f"heading_{level}")
        for level in range(1, 5)
    ]

    # Remove missing heading values
    headings = [
        heading
        for heading in headings
        if heading
    ]

    # Remove navigation sections such as "Next Lesson"
    has_navigation_heading = any(
        normalize_value(heading) in NAVIGATION_HEADINGS
        for heading in headings
    )

    if has_navigation_heading:
        return False

    return True

    

In [134]:
removed_chunks = [
    chunk
    for chunk in chunks
    if not is_useful_chunk(chunk)
]

filtered_chunks = [
    chunk
    for chunk in chunks
    if is_useful_chunk(chunk)
]

print("Chunks before filtering:", len(chunks))
print("Removed chunks:", len(removed_chunks))
print("Chunks after filtering:", len(filtered_chunks))

for chunk in removed_chunks:
    print("=" * 70)
    print("ID:", chunk.metadata.get("chunk_id"))
    print("Lesson:", chunk.metadata.get("lesson"))
    print("Content:", repr(chunk.page_content))

Chunks before filtering: 149
Removed chunks: 14
Chunks after filtering: 135
ID: course-chunk-0014
Lesson: 01-intro-to-ai-agents
Content: '## Previous Lesson  \nCourse Setup'
ID: course-chunk-0015
Lesson: 01-intro-to-ai-agents
Content: '## Next Lesson  \nExploring Agentic Frameworks'
ID: course-chunk-0030
Lesson: 03-agentic-design-patterns
Content: '## Previous Lesson  \nExploring Agentic Frameworks'
ID: course-chunk-0031
Lesson: 03-agentic-design-patterns
Content: '## Next Lesson  \nTool Use Design Pattern'
ID: course-chunk-0059
Lesson: 04-tool-use
Content: '## Previous Lesson  \nUnderstanding Agentic Design Patterns'
ID: course-chunk-0060
Lesson: 04-tool-use
Content: '## Next Lesson  \nAgentic RAG'
ID: course-chunk-0082
Lesson: 05-agentic-rag
Content: '## Previous Lesson  \nTool Use Design Pattern'
ID: course-chunk-0083
Lesson: 05-agentic-rag
Content: '## Next Lesson  \nBuilding Trustworthy AI Agents'
ID: course-chunk-0101
Lesson: 07-planning-design
Content: '## Previous Lesson  \nBui

In [135]:
chunks = filtered_chunks

for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = (
        f"course-chunk-{index:04d}"
    )

chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Final number of chunks:", len(chunks))
print("Smallest chunk length:", min(chunk_lengths))
print("Largest chunk length:", max(chunk_lengths))

Final number of chunks: 135
Smallest chunk length: 56
Largest chunk length: 1200


# Embeddings and Vector Store

Chunks are embedded with dense semantic retrieval using `BAAI/bge-small-en-v1.5`, then stored in Chroma for similarity search.

This project uses dense embeddings only — not dense-and-sparse hybrid search.


In [139]:
from langchain_huggingface import HuggingFaceEmbeddings


EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

print("Embedding model is ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12755.46it/s]


In [150]:
# test_vector = embedding_model.embed_query(
#     "What is an AI agent?"
# )

# print("Vector dimensions:", len(test_vector))
# print("First 5 values:", test_vector[:5])

### Chroma Vector Store

A local Chroma collection holds the embedded chunks. Later retrieval tools run real similarity searches against this store rather than returning hardcoded content.


In [141]:
from langchain_chroma import Chroma
import shutil

In [142]:
print("Vector store directory:")
print(VECTORSTORE_DIR)

Vector store directory:
/Users/khawla/learngraph/data/vectorstore/course


In [ ]:
vectorstore = Chroma(
    collection_name="learngraph_course_bge",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR),
)

print("Empty Chroma vector store created")

Empty Chroma vector store created


In [146]:
chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in chunks
]

added_ids = vectorstore.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("chunks added to Chroma")
print("Number of added IDs:", len(added_ids))

chunks added to Chroma
Number of added IDs: 135


In [147]:
stored_data = vectorstore.get()
stored_count = len(stored_data["ids"])

print("Chunks before indexing:", len(chunks))
print("Chunks stored in Chroma:", stored_count)

Chunks before indexing: 135
Chunks stored in Chroma: 135


In [148]:
stored_sample = vectorstore.get(
    ids=[chunk_ids[0]]
)

print("Stored ID:")
print(stored_sample["ids"][0])

print("\nStored metadata:")
print(stored_sample["metadatas"][0])

print("\nStored document:")
print(stored_sample["documents"][0][:700])

Stored ID:
course-chunk-0000

Stored metadata:
{'start_index': 0, 'source_type': 'microsoft_ai_agents_course', 'relative_path': 'microsoft-ai-agents/01-intro-to-ai-agents.md', 'file_name': '01-intro-to-ai-agents.md', 'title': 'Introduction to AI Agents and Agent Use Cases', 'chunk_id': 'course-chunk-0000', 'source_url': 'https://github.com/microsoft/ai-agents-for-beginners/blob/main/01-intro-to-ai-agents/README.md', 'heading_1': 'Introduction to AI Agents and Agent Use Cases', 'lesson': '01-intro-to-ai-agents'}

Stored document:
# Introduction to AI Agents and Agent Use Cases  
Welcome to the **AI Agents for Beginners** course! This course gives you the foundational knowledge — and real working code — to start building AI Agents from scratch.  
Before we jump into building, let's make sure we actually understand what an AI Agent *is* and when it makes sense to use one.  
---


In [ ]:
tool_results = vectorstore.similarity_search( #test
    query="How can AI agents use tools?",
    k=4,
)

In [153]:
for index, document in enumerate(
    tool_results,
    start=1,
):
    print(
        index,
        document.metadata.get("lesson"),
        "—",
        document.metadata.get("heading_3")
        or document.metadata.get("heading_2")
        or document.metadata.get("heading_1"),
    )

1 04-tool-use — What are the use cases it can be applied to?
2 04-tool-use — Tool Use Design Pattern
3 04-tool-use — Microsoft Foundry Agent Service
4 01-intro-to-ai-agents — When to Use AI Agents


In [155]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
    },
)

print("Course retriever is ready")

Course retriever is ready


In [ ]:
retrieved_documents = retriever.invoke( #test retriever
    "Explain agent memory."
)

for index, document in enumerate(
    retrieved_documents,
    start=1,
):
    print(
        index,
        document.metadata.get("lesson"),
        "—",
        document.metadata.get("heading_3")
        or document.metadata.get("heading_2")
        or document.metadata.get("heading_1"),
    )

1 13-agent-memory — Understanding AI Agent Memory
2 13-agent-memory — Why is Memory Important?
3 13-agent-memory — Introduction
4 13-agent-memory — Implementing and Storing Memory


# Adding a Second Knowledge Source

Official LangGraph documentation is loaded, chunked, and indexed into the **same** Chroma store with a different `source_type`.

That lets the agent route conceptual questions to the course and implementation questions (Functional API, interrupts, checkpoints) to the docs.


In [167]:
LANGGRAPH_DOCS_DIR = (
    PROJECT_ROOT / "data" / "langgraph_docs"
)

LANGGRAPH_SOURCE_URLS = {
    "functional-api.md": (
        "https://docs.langchain.com/oss/python/"
        "langgraph/functional-api"
    ),
    "interrupts.md": (
        "https://docs.langchain.com/oss/python/"
        "langgraph/interrupts"
    ),
}

langgraph_files = sorted(
    LANGGRAPH_DOCS_DIR.glob("*.md")
)

print("LangGraph files:", len(langgraph_files))

for file_path in langgraph_files:
    print("-", file_path.name)

LangGraph files: 2
- functional-api.md
- interrupts.md


In [168]:
langgraph_documents = []

for file_path in langgraph_files:
    cleaned_text = clean_markdown(
        file_path.read_text(
            encoding="utf-8"
        )
    )

    document = Document(
        page_content=cleaned_text,
        metadata={
            "title": file_path.stem.replace(
                "-",
                " ",
            ).title(),
            "file_name": file_path.name,
            "relative_path": str(
                file_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "source_type": "langgraph_docs",
            "source_url": LANGGRAPH_SOURCE_URLS[
                file_path.name
            ],
        },
    )

    langgraph_documents.append(document)

print(
    "LangGraph documents:",
    len(langgraph_documents),
)


LangGraph documents: 2


In [169]:
langgraph_sections = (
    split_documents_by_headers(
        langgraph_documents
    )
)

langgraph_chunks = (
    recursive_splitter.split_documents(
        langgraph_sections
    )
)

for index, chunk in enumerate(
    langgraph_chunks
):
    chunk.metadata["chunk_id"] = (
        f"langgraph-chunk-{index:04d}"
    )

print(
    "LangGraph sections:",
    len(langgraph_sections),
)

print(
    "LangGraph chunks:",
    len(langgraph_chunks),
)

LangGraph sections: 48
LangGraph chunks: 83


In [171]:
stored_ids = vectorstore.get()["ids"]

old_langgraph_ids = [
    document_id
    for document_id in stored_ids
    if document_id.startswith(
        "langgraph-chunk-"
    )
]

if old_langgraph_ids:
    vectorstore.delete(
        ids=old_langgraph_ids
    )

langgraph_chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in langgraph_chunks
]

vectorstore.add_documents(
    documents=langgraph_chunks,
    ids=langgraph_chunk_ids,
)

print(
    "LangGraph chunks added:",
    len(langgraph_chunk_ids),
)

LangGraph chunks added: 83


In [172]:
stored_data = vectorstore.get()

source_counts = {}

for metadata in stored_data["metadatas"]:
    source_type = metadata.get(
        "source_type",
        "unknown",
    )

    source_counts[source_type] = (
        source_counts.get(
            source_type,
            0,
        )
        + 1
    )

print(source_counts)

{'microsoft_ai_agents_course': 135, 'langgraph_docs': 83}


In [173]:
langgraph_results = (
    vectorstore.similarity_search(
        query=(
            "How does interrupt pause "
            "and resume a LangGraph workflow?"
        ),
        k=4,
        filter={
            "source_type": "langgraph_docs"
        },
    )
)

for index, document in enumerate(
    langgraph_results,
    start=1,
):
    print(
        f"\n--- RESULT {index} ---"
    )
    print(
        "Page:",
        document.metadata.get("title"),
    )
    print(
        document.page_content[:700]
    )


--- RESULT 1 ---
Page: Interrupts
## Pause using `interrupt`  
The `interrupt` function pauses graph execution and returns a value to the caller. When you call `interrupt` within a node, LangGraph saves the current graph state and waits for you to resume execution with input.  
To use `interrupt`, you need:  
1. A **checkpointer** to persist the graph state (use a durable checkpointer in production)
2. A **thread ID** in your config so the runtime knows which state to resume from
3. To call `interrupt()` where you want to pause (payload must be JSON-serializable)  
```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langgraph.types import interrupt

def approval_node(state: State):
# Pause a

--- RESULT 2 ---
Page: Interrupts
# Interrupts  
Interrupts allow you to pause graph execution at specific points and wait for external input before continuing. This enables human-in-the-loop patterns where you need external input to proceed. When an interrupt is

# Retrieval Tools and Multi-Source Routing

The agent selects one or both retrieval tools depending on the question:

- `search_course` — AI-agent concepts from the Microsoft course
- `search_langgraph_docs` — LangGraph implementation details
- **Both** — mixed conceptual + implementation questions

Each tool runs a real Chroma similarity search filtered by `source_type`.


In [191]:
from langchain.tools import tool

@tool
def search_course(query: str) -> str:
    """
    Search the Microsoft AI Agents course.

    Use this tool for conceptual questions about
    AI agents, tools, RAG, planning, context,
    and agent memory.
    """

    documents = vectorstore.similarity_search(
        query=query,
        k=4,
        filter={
            "source_type":
            "microsoft_ai_agents_course"
        },
    )

    context_parts = []

    for index, document in enumerate(
        documents,
        start=1,
    ):
        metadata = document.metadata

        context_parts.append(
            f"""
[COURSE RESULT {index}]

Lesson: {metadata.get("lesson", "Unknown")}
Source: {metadata.get("source_url", "Unknown")}

Content:
{document.page_content}
""".strip()
        )

    return "\n\n---\n\n".join(
        context_parts
    )

In [192]:
@tool
def search_langgraph_docs(
    query: str,
) -> str:
    """
    Search the official LangGraph documentation.

    Use this tool for implementation questions about
    the Functional API, tasks, entrypoints, interrupts,
    checkpoints, thread IDs, and workflow resuming.
    """

    documents = vectorstore.similarity_search(
        query=query,
        k=4,
        filter={
            "source_type": "langgraph_docs"
        },
    )

    context_parts = []

    for index, document in enumerate(
        documents,
        start=1,
    ):
        metadata = document.metadata

        context_parts.append(
            f"""
[LANGGRAPH RESULT {index}]

Page: {metadata.get("title", "Unknown")}
Source: {metadata.get("source_url", "Unknown")}

Content:
{document.page_content}
""".strip()
        )

    return "\n\n---\n\n".join(
        context_parts
    )

In [193]:
# course_test = search_course.invoke(
#     {
#         "query": "What is agent memory?"
#     }
# )

# print(course_test[:1500])


langgraph_test = (
    search_langgraph_docs.invoke(
        {
            "query": (
                "How does interrupt pause "
                "a LangGraph workflow?"
            )
        }
    )
)

print(langgraph_test[:1500])

[LANGGRAPH RESULT 1]

Page: Interrupts
Source: https://docs.langchain.com/oss/python/langgraph/interrupts

Content:
## Pause using `interrupt`  
The `interrupt` function pauses graph execution and returns a value to the caller. When you call `interrupt` within a node, LangGraph saves the current graph state and waits for you to resume execution with input.  
To use `interrupt`, you need:  
1. A **checkpointer** to persist the graph state (use a durable checkpointer in production)
2. A **thread ID** in your config so the runtime knows which state to resume from
3. To call `interrupt()` where you want to pause (payload must be JSON-serializable)  
```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langgraph.types import interrupt

def approval_node(state: State):
# Pause and ask for approval
approved = interrupt("Do you approve this action?")

---

[LANGGRAPH RESULT 2]

Page: Interrupts
Source: https://docs.langchain.com/oss/python/langgraph/interrupts


In [194]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY was not found in the .env file."
    )

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    timeout=60,
    max_retries=2,
)

print("OpenAI model is ready")

OpenAI model is ready


In [195]:
search_course
search_langgraph_docs

StructuredTool(name='search_langgraph_docs', description='Search the official LangGraph documentation.\n\nUse this tool for implementation questions about\nthe Functional API, tasks, entrypoints, interrupts,\ncheckpoints, thread IDs, and workflow resuming.', args_schema=<class 'langchain_core.utils.pydantic.search_langgraph_docs'>, func=<function search_langgraph_docs at 0x145ec54e0>)

In [196]:
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
)

from langgraph.func import (
    entrypoint,
    task,
)


tools = [
    search_course,
    search_langgraph_docs,
]

tools_by_name = {
    current_tool.name: current_tool
    for current_tool in tools
}


routing_llm = llm.bind_tools(
    tools,
    tool_choice="any",
)

In [208]:
ROUTER_PROMPT = """
You are the source router for LearnGraph.

Read the student's question and call the most
appropriate retrieval tool.

Available tools:

1. search_course
Use for conceptual questions about AI agents,
tools, agentic RAG, planning, context engineering,
and agent memory.

2. search_langgraph_docs
Use for implementation questions about LangGraph,
@task, @entrypoint, interrupt(), checkpoints,
thread IDs, and workflow resuming.

Call both tools when the question combines a general
AI-agent concept with LangGraph implementation.

Do not answer the student's question yet.
Only create the required tool call or tool calls.
""".strip()


ANSWER_PROMPT = """
You are LearnGraph, an educational tutor.

Answer the student's question using only the content
returned by the retrieval tools.

Rules:

1. Explain clearly for a beginner.
2. Do not add unsupported information.
3. Mention the source URLs included in the tool results.
4. If the retrieved content is insufficient, say so.
5. Do not invent sources.
""".strip()

## State and Memory

- **`InMemorySaver`** with a stable `thread_id` — short-term conversation state so follow-up questions work in the same thread
- **`InMemoryStore`** — learner quiz progress (topic, answer, score, feedback)

Both are in-memory, so they clear when the kernel restarts.


In [209]:
from pydantic import BaseModel, Field

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
)
from langchain_core.runnables import RunnableConfig

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.graph import add_messages
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langgraph.types import (
    Command,
    RetryPolicy,
    interrupt,
)


# Short-term memory and workflow checkpoints
checkpointer = InMemorySaver()

# Long-term student progress
memory_store = InMemoryStore()

# Retry temporary network/API failures
retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=1.0,
)

print(" Memory and retry policy are ready")

 Memory and retry policy are ready


In [210]:
class RelevanceGrade(BaseModel):
    relevant: bool = Field(
        description=(
            "True when the retrieved content can answer "
            "the student's question."
        )
    )

    reason: str = Field(
        description="A short explanation of the decision."
    )


relevance_llm = llm.with_structured_output(
    RelevanceGrade
)

# Agentic RAG Agent

LearnGraph uses **Agentic RAG with multi-source routing and corrective retrieval**.

The agent must:

1. Choose the appropriate knowledge source
2. Retrieve relevant content
3. Evaluate retrieval quality
4. Rewrite weak queries and retrieve again when needed


In [211]:
@task(retry_policy=retry_policy)
def route_question(
    messages: list[BaseMessage],
):
    """Select one or both retrieval tools."""

    return routing_llm.invoke(
        [
            SystemMessage(
                content=ROUTER_PROMPT
            ),
            *messages,
        ]
    )


@task(retry_policy=retry_policy)
def execute_tool(
    tool_call: dict,
):
    """Execute a selected retrieval tool."""

    selected_tool = tools_by_name[
        tool_call["name"]
    ]

    return selected_tool.invoke(
        tool_call
    )


@task(retry_policy=retry_policy)
def grade_retrieval(
    question: str,
    tool_results: list[BaseMessage],
) -> dict:
    """Check whether retrieved content is relevant."""

    context = "\n\n".join(
        str(result.content)
        for result in tool_results
    )

    grade = relevance_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Evaluate whether the retrieved context "
                    "contains enough relevant information to "
                    "answer the student's question. Be strict."
                )
            ),
            HumanMessage(
                content=f"""
Student question:

{question}

Retrieved context:

{context[:8000]}
""".strip()
            ),
        ]
    )

    return grade.model_dump()


@task(retry_policy=retry_policy)
def rewrite_search_query(
    question: str,
    failed_context: str,
) -> str:
    """Rewrite a weak retrieval query."""

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Rewrite the student's question into one "
                    "short and focused English search query. "
                    "Return only the rewritten query."
                )
            ),
            HumanMessage(
                content=f"""
Original question:

{question}

The previous retrieval was not sufficient.

Previous context sample:

{failed_context[:2000]}
""".strip()
            ),
        ]
    )

    return response.content.strip()


@task(retry_policy=retry_policy)
def generate_grounded_answer(
    messages: list[BaseMessage],
):
    """Generate the final grounded answer."""

    return llm.invoke(
        [
            SystemMessage(
                content=ANSWER_PROMPT
            ),
            *messages,
        ]
    )

In [212]:
@entrypoint(
    checkpointer=checkpointer,
    store=memory_store,
)
def learn_agent(
    inputs: list[BaseMessage],
    *,
    previous: list[BaseMessage] | None = None,
):
    """
    Multi-source Agentic RAG with:
    - routing
    - relevance grading
    - query rewriting
    - short-term memory
    """

    # Restore this thread's previous conversation
    conversation = add_messages(
        previous or [],
        inputs,
    )

    latest_question = next(
        (
            message.content
            for message in reversed(inputs)
            if isinstance(message, HumanMessage)
        ),
        "",
    )

    search_messages = conversation

    final_routing_response = None
    final_tool_results = []
    final_grade = None
    used_query = latest_question

    maximum_attempts = 2

    for attempt in range(
        1,
        maximum_attempts + 1,
    ):
        # 1. Choose source or sources
        routing_response = route_question(
            search_messages
        ).result()

        if not routing_response.tool_calls:
            raise RuntimeError(
                "The agent did not select a retrieval tool."
            )

        # 2. Execute selected retrieval tools
        tool_futures = [
            execute_tool(tool_call)
            for tool_call
            in routing_response.tool_calls
        ]

        tool_results = [
            future.result()
            for future in tool_futures
        ]

        # 3. Grade retrieved content
        grade = grade_retrieval(
            latest_question,
            tool_results,
        ).result()

        final_routing_response = routing_response
        final_tool_results = tool_results
        final_grade = grade

        if grade["relevant"]:
            break

        # Do not rewrite after the final attempt
        if attempt == maximum_attempts:
            break

        failed_context = "\n\n".join(
            str(result.content)
            for result in tool_results
        )

        # 4. Rewrite and loop back
        used_query = rewrite_search_query(
            latest_question,
            failed_context,
        ).result()

        search_messages = add_messages(
            conversation,
            HumanMessage(
                content=(
                    "Use this focused retrieval query: "
                    f"{used_query}"
                )
            ),
        )

    # 5. Give the LLM the successful/final context
    messages_with_context = add_messages(
        conversation,
        [
            final_routing_response,
            *final_tool_results,
        ],
    )

    final_response = generate_grounded_answer(
        messages_with_context
    ).result()

    saved_conversation = add_messages(
        messages_with_context,
        final_response,
    )

    # Return result to user, but save conversation in checkpoint
    return entrypoint.final(
        value={
            "selected_tools": [
                tool_call["name"]
                for tool_call
                in final_routing_response.tool_calls
            ],
            "search_query": used_query,
            "retrieval_relevant": final_grade[
                "relevant"
            ],
            "relevance_reason": final_grade[
                "reason"
            ],
            "answer": final_response.content,
        },
        save=saved_conversation,
    )

In [213]:
learn_config = {
    "configurable": {
        "thread_id": "learn-thread-1",
        "user_id": "student-1",
    }
}


learn_result = learn_agent.invoke(
    [
        HumanMessage(
            content=(
                "How does an AI agent remember "
                "previous interactions?"
            )
        )
    ],
    config=learn_config,
)

print("Tools:", learn_result["selected_tools"])
print("Relevant:", learn_result["retrieval_relevant"])
print("Reason:", learn_result["relevance_reason"])
print("\nAnswer:")
print(learn_result["answer"])

Tools: ['search_course']
Relevant: True
Reason: The retrieved context explains what AI agent memory is, why it is important, and how it is implemented, including mechanisms for generating, storing, retrieving, and updating information. It also provides an example of entity memory, showing how an agent can remember specific details from past interactions. This information sufficiently answers how an AI agent remembers previous interactions.

Answer:
An AI agent remembers previous interactions through a concept called "memory," which allows it to retain and recall information such as conversation details, user preferences, past actions, or learned patterns. Without memory, the AI would treat each interaction as new and unrelated, leading to a poor user experience.

Memory in AI agents is managed through processes like generating, storing, retrieving, integrating, updating, and sometimes deleting information. One specific type of memory is "Entity Memory," where the agent extracts and rem

In [214]:
follow_up_result = learn_agent.invoke(
    [
        HumanMessage(
            content=(
                "Can you explain that again "
                "using a simpler example?"
            )
        )
    ],
    config=learn_config,
)

print(follow_up_result["answer"])

Sure! Here's a simpler example to explain how an AI agent remembers previous interactions:

Imagine you are talking to a helpful robot friend. The first time you tell the robot, "I like chocolate ice cream," the robot remembers this fact. Later, when you ask the robot, "What ice cream do I like?" it can answer, "You like chocolate ice cream," because it remembered what you said before.

This memory works by the robot saving important details from your conversation (like "chocolate ice cream") and then retrieving that information when needed. Without this memory, the robot would forget what you told it and would not be able to give you personalized answers.

So, AI agent memory is like the robot's notebook where it writes down important things you say, so it can remember and use them in future talks.

This explanation is based on the concept of AI agent memory described here: https://github.com/microsoft/ai-agents-for-beginners/blob/main/13-agent-memory/README.md


# Quiz Mode and Human-in-the-Loop

Quiz Mode uses `interrupt()` to pause after generating a question. The workflow waits for the student's answer and resumes with:

```python
Command(resume=student_answer)
```

Progress is then saved in `InMemoryStore` for the learner.


In [215]:
class QuizQuestion(BaseModel):
    question: str = Field(
        description="One clear beginner-level quiz question."
    )

    reference_answer: str = Field(
        description=(
            "The correct answer based only on "
            "the retrieved course context."
        )
    )


class QuizGrade(BaseModel):
    score: int = Field(
        ge=0,
        le=100,
    )

    correct: bool

    feedback: str = Field(
        description=(
            "Short constructive feedback for the student."
        )
    )

In [216]:
quiz_question_llm = llm.with_structured_output(
    QuizQuestion
)

quiz_grade_llm = llm.with_structured_output(
    QuizGrade
)

In [217]:
@task(retry_policy=retry_policy)
def generate_quiz_question(
    topic: str,
) -> dict:
    """Retrieve course content and create one quiz."""

    course_context = search_course.invoke(
        {
            "query": topic
        }
    )

    quiz = quiz_question_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Create one beginner-level quiz question "
                    "using only the provided course context."
                )
            ),
            HumanMessage(
                content=f"""
Topic:

{topic}

Course context:

{course_context[:8000]}
""".strip()
            ),
        ]
    )

    return {
        "question": quiz.question,
        "reference_answer": quiz.reference_answer,
        "context": course_context,
    }


@task(retry_policy=retry_policy)
def evaluate_quiz_answer(
    question: str,
    reference_answer: str,
    student_answer: str,
    context: str,
) -> dict:
    """Evaluate the student's quiz answer."""

    grade = quiz_grade_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Evaluate the student's answer using only "
                    "the reference answer and course context. "
                    "Be fair and provide beginner-friendly feedback."
                )
            ),
            HumanMessage(
                content=f"""
Question:

{question}

Reference answer:

{reference_answer}

Student answer:

{student_answer}

Course context:

{context[:6000]}
""".strip()
            ),
        ]
    )

    return grade.model_dump()

In [218]:
@entrypoint(
    checkpointer=checkpointer,
    store=memory_store,
)
def quiz_agent(
    inputs: dict,
    *,
    store: BaseStore,
    config: RunnableConfig,
):
    """
    Generate a quiz, pause for the student,
    evaluate the answer, and save progress.
    """

    topic = inputs["topic"]

    # This task result is checkpointed
    quiz = generate_quiz_question(
        topic
    ).result()

    # Real human-in-the-loop pause
    student_answer = interrupt(
        {
            "type": "quiz_question",
            "topic": topic,
            "question": quiz["question"],
        }
    )

    grade = evaluate_quiz_answer(
        question=quiz["question"],
        reference_answer=quiz[
            "reference_answer"
        ],
        student_answer=student_answer,
        context=quiz["context"],
    ).result()

    user_id = config["configurable"].get(
        "user_id",
        "student-1",
    )

    thread_id = config["configurable"][
        "thread_id"
    ]

    # Long-term memory: progress can be read
    # independently from the conversation thread
    store.put(
        (
            "learners",
            user_id,
            "quiz_progress",
        ),
        thread_id,
        {
            "topic": topic,
            "question": quiz["question"],
            "student_answer": student_answer,
            "score": grade["score"],
            "correct": grade["correct"],
            "feedback": grade["feedback"],
        },
    )

    return {
        "topic": topic,
        "question": quiz["question"],
        "student_answer": student_answer,
        "score": grade["score"],
        "correct": grade["correct"],
        "feedback": grade["feedback"],
    }

In [219]:
quiz_config = {
    "configurable": {
        "thread_id": "quiz-thread-1",
        "user_id": "student-1",
    }
}


quiz_started = quiz_agent.invoke(
    {
        "topic": "agent memory"
    },
    config=quiz_config,
)

quiz_interrupt = quiz_started[
    "__interrupt__"
][0]

print("Quiz question:")
print(quiz_interrupt.value["question"])

Quiz question:
What is the primary purpose of memory in AI agents?


In [220]:
quiz_result = quiz_agent.invoke(
    Command(
        resume=(
            "Agent memory allows the agent "
            "to retain and reuse information "
            "from previous interactions."
        )
    ),
    config=quiz_config,
)

print("Score:", quiz_result["score"])
print("Correct:", quiz_result["correct"])
print("Feedback:", quiz_result["feedback"])

Score: 8
Correct: True
Feedback: Your answer correctly identifies that agent memory allows retention and reuse of information from previous interactions, which aligns well with the primary purpose of memory in AI agents. To improve, you could mention that this memory enables better and more personalized interactions by recalling details like user preferences or learned patterns, as highlighted in the course material.


In [221]:
saved_progress = memory_store.get(
    (
        "learners",
        "student-1",
        "quiz_progress",
    ),
    "quiz-thread-1",
)

print(saved_progress.value)

{'topic': 'agent memory', 'question': 'What is the primary purpose of memory in AI agents?', 'student_answer': 'Agent memory allows the agent to retain and reuse information from previous interactions.', 'score': 8, 'correct': True, 'feedback': 'Your answer correctly identifies that agent memory allows retention and reuse of information from previous interactions, which aligns well with the primary purpose of memory in AI agents. To improve, you could mention that this memory enables better and more personalized interactions by recalling details like user preferences or learned patterns, as highlighted in the course material.'}
